# M3 — 메인 LangGraph에 전문 Tool 4개 연결, Tool별 1-Tool 실행 로그 (11/05)

- 일정표 11/05 M3: **Tool 3개·메인 Graph 최초 실행 — 각 ToolResult + 메인 1-Tool QueryOutput**.
- 연결 방식: 메인 그래프(`src/rag/school/main_graph.py`)는 다른 담당자 모듈을 import하지 않는다. 이 노트북이 registry
  `{Tool 이름: callable}`로 넘긴다(슬러그 폴더 간 import 금지 규칙). 드라이버는 이 노트북이 한 번 열고 마지막에 닫는다.
- Tool 코드(팀원 push, 이 노트북에서 수정하지 않음):
  - `action_search` — 주승우 `src/rag/school/action_tool.py`
  - `disease_search`, `crossdoc_graph_search` — 송수림 `src/rag/respiratory/disease_tool.py`, `crossdoc_tool.py`
  - `legal_search` — 유종혁 `src/rag/prevention_legal/tool.py` (**현재 고정 샘플 질문 1개만 답하는 단계**, run.py 5행).
    로컬 파일 `data/interim/prevention_legal/blocks.jsonl`(Git 제외, 종혁님 노트북 02가 생성)이 있어야 실행된다.
    파일이 없으면 Tool이 ERROR(FileNotFoundError)를 돌려주고 메인은 호출 1회·ERROR로 기록한다.
- 각 실행에서 확인: ① 메인이 고른 Tool ② 실제 호출 1회 ③ ToolResult 계약 검사 ④ 최종 QueryOutput 계약 검사
  ⑤ 근거가 있으면 최종 출력의 Citation·Finding을 Neo4j에서 다시 대조.

In [1]:
import json
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)

from index.school.dictionaries import load_dictionaries
from rag.school.contract_v2_local import validate_query_output, validate_tool_result
from rag.school.graph import default_deps
from rag.school.main_graph import default_registry, run_main
from rag.school.validation_local import verify_against_db
# 팀원 Tool — registry로만 넘긴다(메인 모듈 안에서 import하지 않음)
from rag.respiratory.disease_tool import disease_search
from rag.respiratory.crossdoc_tool import crossdoc_graph_search
from rag.prevention_legal.tool import legal_search
from rag.prevention_legal.run import SAMPLE_QUESTION as LEGAL_SAMPLE

deps = default_deps()
deps["books"] = load_dictionaries()
DRIVER, DATABASE = deps["driver"], deps["database"]
ALLOWED = ["moe_school_manual_v3", "gyeonggi_school_guide_2025", "kdca_measles_2026",
           "kdca_diagnosis_reporting_2026", "moleg_infectious_disease_act_21521"]
REGISTRY = {**default_registry(deps),
            "disease_search": lambda ti: disease_search(ti, deps=deps),       # 공유 드라이버 사용(닫지 않음)
            "crossdoc_graph_search": lambda ti: crossdoc_graph_search(ti, deps=deps),
            "legal_search": legal_search}                                    # 자체 연결을 열고 닫는다
print("연결한 Tool:", sorted(REGISTRY))

연결한 Tool: ['action_search', 'crossdoc_graph_search', 'disease_search', 'legal_search']


## Tool별 1-Tool 실행 (실DB)

In [2]:
CASES = [("action_search", "학교에서 유증상 학생이 발견되면 담임교사와 보건교사는 각각 무엇을 확인하나요?"),
         ("disease_search", "홍역의 잠복기는 며칠인가요?"),
         ("crossdoc_graph_search", "보호자에게 연락해 진료를 요청하는 일을 교육부 매뉴얼과 경기도 가이드북은 각각 어떻게 설명하나요?"),
         ("legal_search", LEGAL_SAMPLE),                                   # 고정 샘플 질문
         ("legal_search", "성홍열도 학교장이 보건소에 신고해야 하나요?")]      # 샘플 밖 질문(근거 없음이 정상)

logs, outputs = [], {}
for n, (expected, question) in enumerate(CASES, 1):
    query = {"question": question, "region": "GYEONGGI", "allowed_document_ids": ALLOWED, "request_id": f"m3-{n:02d}"}
    started = time.perf_counter()
    out = run_main(query, deps, REGISTRY)
    seconds = round(time.perf_counter() - started, 1)
    validate_query_output(out, query)
    trace = deps["main_trace"]
    selected = next((t.get("tool") for t in trace if t["step"] == "select_tool"), None)
    executed = [t for t in trace if t["step"] == "execute_tool"]
    results = trace[-1]["results"]
    tool_input = {"question": question, "task": "-", "region": "GYEONGGI", "allowed_document_ids": ALLOWED}
    contract = "통과"
    for result in results:
        try:
            validate_tool_result(result, tool_input, selected)
        except Exception as exc:  # noqa: BLE001
            contract = f"위반: {exc}"
    db = None
    if out["findings"]:
        report = verify_against_db(out, DRIVER, DATABASE)
        db = "통과" if not report["bad_citations"] and not report["bad_findings"] else str(report)[:80]
    outputs[f"m3-{n:02d}"] = out
    logs.append({"request_id": f"m3-{n:02d}", "기대 Tool": expected, "선택 Tool": selected,
                 "호출": out["search_rounds"], "tools_used": out["tools_used"],
                 "Tool status": executed[0]["status"] if executed else None,
                 "ToolResult 계약": contract if results else "-", "최종 status": out["status"],
                 "Finding": len(out["findings"]), "Citation": len(out["citations"]),
                 "GraphPath(ToolResult)": sum(len(r["graph_paths"]) for r in results),
                 "최종 출력 DB 재대조": db, "초": seconds})
log_table = pd.DataFrame(logs)
display(log_table)

,request_id,기대 Tool,선택 Tool,호출,tools_used,Tool status,ToolResult 계약,최종 status,Finding,Citation,GraphPath(ToolResult),최종 출력 DB 재대조,초
0,m3-01,action_search,action_search,1,[action_search],OK,통과,ANSWERED,12,11,8,통과,4.4
1,m3-02,disease_search,disease_search,1,[disease_search],OK,통과,ANSWERED,4,4,4,통과,2.1
2,m3-03,crossdoc_graph_search,crossdoc_graph_search,1,[crossdoc_graph_search],OK,통과,ANSWERED,11,7,11,통과,1.9
3,m3-04,legal_search,legal_search,1,[legal_search],ERROR,통과,ERROR,0,0,0,NaN,0.0
4,m3-05,legal_search,legal_search,1,[legal_search],NO_EVIDENCE,통과,INSUFFICIENT_EVIDENCE,0,0,0,NaN,0.0


### 답변(서비스가 보여 주는 형태)

In [3]:
for rid, out in outputs.items():
    print(f"=== {rid} | {out['tools_used']} | {out['status']}")
    print(out["answer"])
    print()

=== m3-01 | ['action_search'] | ANSWERED
원문에서 확인한 내용입니다.

■ 담임교사
- 감염병 유증상자를 발견한다.  (경기도 p.166 · 대응 1단계) [1]
- 의료기관 진료 결과를 확인하고 조치한다.  (경기도 p.166 · 대응 1단계) [1]
- 결석·조퇴·지각 사유를 확인하고 의심환자를 파악한다.  (교육부 p.43(시나리오 예시)) [5]

■ 보건교사
- 환자의 증상과 징후를 관찰하고 감염병이 의심되는지를 확인한다.  (교육부 p.36(시나리오 예시) · 대응 1단계) [2]
- 감염병을 의심할 수 있는 증상인지 확인한다.  (경기도 p.167 · 대응 1단계) [3]
- 일일 보건실 이용 학생의 감염병 증상 여부를 확인한다. (시점: 일일)  (교육부 p.42(시나리오 예시)) [4]

■ 담당이 원문에 적혀 있지 않은 행동
- 감염병(의심) 환자 발생 여부를 확인한다. (조건: 결과를 확인 후)  (경기도 p.164 · 대응 1단계) [6]
- 학생을 최대한 안정시킨 상태에서 증상 변화를 주의 깊게 관찰한다.  (교육부 p.56 · 대응 1단계) [7]

■ 원문 그대로 (행동으로 정리되지 않은 문단·표)
- “감염병 (의심)환자가 교내에서 혼자 이동하는 것은 원칙적으로 금지 이동 시에는 담임(담당) 교사가 동행하며 2m 정도의 거리 유지 담당 교사는 담임교사 권장 (만약 임신부 등 감염병 고위험군인 경우 다른 교사로 대체)”  (교육부 p.54 · 대응 1단계) [8]
- “7) 교사 및 학부모의 역할 ∙ 정기적인 예방접종 권장 ∙ 보건(담당)교사 및 담임교사는 학교 내 (의심)환자 발생에 따라 교육(지원)청에 보고 및 조치함”  (교육부 p.197) [9]

그 밖의 근거 2건은 findings·citations에 있습니다.

참고: 원문 인용 Finding은 검색된 원문을 그대로 옮긴 것이다. 해석·적용 판단은 포함하지 않는다. / 역할은 원문이 제시한 예시이며 학교 상황에 따라 조정할 수 있다(의무 배

### M3 판정

In [4]:
ok = log_table[(log_table["호출"] == 1) & (log_table["선택 Tool"] == log_table["기대 Tool"])
                & (log_table["ToolResult 계약"] == "통과")]
success = ok[ok["최종 status"].isin(["ANSWERED", "PARTIAL"])]
print("Tool별 1-Tool 성공(근거 있는 최종 출력):", sorted(set(success["선택 Tool"])))
print("최대 호출 수:", int(log_table["호출"].max()))
assert {"action_search", "disease_search", "crossdoc_graph_search"} <= set(success["선택 Tool"])
assert (success["최종 출력 DB 재대조"] == "통과").all()
assert (log_table["선택 Tool"] == log_table["기대 Tool"]).all() and (log_table["호출"] == 1).all()
legal = log_table[log_table["기대 Tool"] == "legal_search"][["request_id", "Tool status", "최종 status"]]
print("legal_search: 메인 연결·라우팅·계약 검사는 통과. 근거 있는 성공 로그는 아래 상태 참고")
display(legal)
for rid in legal["request_id"]:
    print(rid, outputs[rid]["warnings"])
out_path = ROOT / "data/interim/school/m3_tool_logs_20261105.json"
out_path.write_text(json.dumps({"logs": logs, "outputs": outputs}, ensure_ascii=False, indent=1), encoding="utf-8")
print("저장:", out_path.relative_to(ROOT))

Tool별 1-Tool 성공(근거 있는 최종 출력): ['action_search', 'crossdoc_graph_search', 'disease_search']
최대 호출 수: 1
legal_search: 메인 연결·라우팅·계약 검사는 통과. 근거 있는 성공 로그는 아래 상태 참고


,request_id,Tool status,최종 status
3,m3-04,ERROR,ERROR
4,m3-05,NO_EVIDENCE,INSUFFICIENT_EVIDENCE


m3-04 ['Tool 실행 실패: FileNotFoundError']
m3-05 ['이 질문은 고정 샘플 범위 밖이거나, 샘플 근거 문서가 허용 목록에 없습니다.']
저장: data\interim\school\m3_tool_logs_20261105.json


In [5]:
DRIVER.close()